# Socratic-OT | Phase 3: Session Memory + Assessment Module

**Project:** Socratic-OT Multimodal AI Tutor  
**Team:** Vidhyadhari Bandaru, Richie Ilavarapu

### What this notebook does
1. **Session Memory Store** — tracks weak topics, repeated confusion, and wrong guesses across a 15-minute session
2. **Proactive Revisiting** — at the start of a new topic within a session, reintroduces previous weak areas
3. **Assessment Module** — prompts the student with a clinical OT scenario and evaluates their reasoning
4. **Mastery Summary** — generates a per-concept and per-session mastery report
5. **Student Dashboard** — bonus: a weak-spot tracker across sessions

### Prerequisite
Run `01_knowledge_base_chromadb.ipynb` and `02_tutoring_policy.ipynb` first.

---
## Step 0 — Install & Setup

In [ ]:
!pip install -q langgraph langchain langchain-groq chromadb sentence-transformers groq

In [ ]:
import os, json, pickle
from datetime import datetime
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/Socratic_OT'
CHROMA_PERSIST     = os.path.join(DRIVE_PROJECT_ROOT, 'Data/chroma_db')
IMAGE_META_JSON    = os.path.join(DRIVE_PROJECT_ROOT, 'Data/image_metadata/image_metadata.json')
MEMORY_DIR         = os.path.join(DRIVE_PROJECT_ROOT, 'Data/session_memory')
os.makedirs(MEMORY_DIR, exist_ok=True)

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = 'gsk_YOUR_KEY_HERE'
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('✅ Setup complete')

In [ ]:
# Reload retriever + LLM (same as previous notebooks)
import chromadb, torch
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

client     = chromadb.PersistentClient(path=CHROMA_PERSIST)
collection = client.get_collection('socratic_ot_textbook')

llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0.4, max_tokens=512)

def retrieve(query, top_k=5):
    qemb = embedder.encode(query, normalize_embeddings=True).tolist()
    res  = collection.query(query_embeddings=[qemb], n_results=top_k,
                             include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist, cid in zip(res['documents'][0], res['metadatas'][0],
                                     res['distances'][0], res['ids'][0]):
        hits.append({'chunk_id': cid, 'score': round(1-dist,4),
                     'chapter': meta.get('chapter',''), 'section': meta.get('section',''),
                     'topic': meta.get('topic',''), 'text': doc})
    merged = {}
    for h in hits:
        k = h['section']
        if k not in merged: merged[k] = h.copy()
        else:
            if h['score'] > merged[k]['score']: merged[k]['score'] = h['score']
            if h['text'] not in merged[k]['text']: merged[k]['text'] += '\n\n' + h['text']
    return sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:3]

print(f'✅ Components loaded | {collection.count()} chunks in ChromaDB')

---
## Step 1 — Session Memory Store

Persisted to disk (Google Drive) so memory survives between Colab sessions.

In [ ]:
class SessionMemoryStore:
    """
    Tracks student performance across a tutoring session.

    Stores:
      - weak_topics      : topics where student needed full reveal
      - confused_terms   : specific terms missed multiple times
      - wrong_guesses    : all incorrect attempts (topic → list of guesses)
      - mastered_topics  : topics where student answered by Turn 2
      - session_timeline : timestamped log of all interactions
    """

    def __init__(self, student_id: str = 'student_001'):
        self.student_id    = student_id
        self.session_start = datetime.now().isoformat()
        self.session_id    = f"{student_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        self.weak_topics     : list  = []
        self.confused_terms  : dict  = {}   # term → count of missed attempts
        self.wrong_guesses   : dict  = {}   # topic → list of wrong guesses
        self.mastered_topics : list  = []
        self.session_timeline: list  = []
        self.topic_scores    : dict  = {}   # topic → {turns_to_correct, needed_reveal}

    # ──────────────────────────────────────────────────────────────────────
    def record_attempt(self, topic: str, guess: str, is_correct: bool, turn_number: int):
        """Record one student attempt for a topic."""
        if topic not in self.wrong_guesses:
            self.wrong_guesses[topic] = []

        if not is_correct:
            self.wrong_guesses[topic].append(guess)
            # Track confusion for specific terms
            self.confused_terms[guess] = self.confused_terms.get(guess, 0) + 1

        self.session_timeline.append({
            'timestamp':  datetime.now().isoformat(),
            'topic':      topic,
            'guess':      guess,
            'correct':    is_correct,
            'turn':       turn_number
        })

    def record_topic_outcome(self, topic: str, turns_to_correct: int, needed_reveal: bool):
        """Record final outcome for a Socratic loop on a topic."""
        self.topic_scores[topic] = {
            'turns_to_correct': turns_to_correct,
            'needed_reveal':    needed_reveal
        }
        if needed_reveal and topic not in self.weak_topics:
            self.weak_topics.append(topic)
        elif not needed_reveal and topic not in self.mastered_topics:
            self.mastered_topics.append(topic)

    # ──────────────────────────────────────────────────────────────────────
    def get_weak_topics_for_revisit(self, n: int = 2) -> list:
        """Return the top-n weak topics to proactively revisit."""
        return self.weak_topics[-n:]

    def get_most_confused_terms(self, n: int = 3) -> list:
        """Return the n most frequently missed terms."""
        sorted_terms = sorted(self.confused_terms.items(), key=lambda x: x[1], reverse=True)
        return [term for term, _ in sorted_terms[:n]]

    # ──────────────────────────────────────────────────────────────────────
    def save(self, save_dir: str = MEMORY_DIR):
        """Persist memory to JSON on Google Drive."""
        data = {
            'student_id':       self.student_id,
            'session_id':       self.session_id,
            'session_start':    self.session_start,
            'weak_topics':      self.weak_topics,
            'confused_terms':   self.confused_terms,
            'wrong_guesses':    self.wrong_guesses,
            'mastered_topics':  self.mastered_topics,
            'topic_scores':     self.topic_scores,
            'session_timeline': self.session_timeline
        }
        path = os.path.join(save_dir, f'{self.session_id}.json')
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'✅ Session memory saved: {path}')
        return path

    @staticmethod
    def load_student_history(student_id: str, save_dir: str = MEMORY_DIR) -> list:
        """Load all past session files for a student."""
        sessions = []
        for fname in os.listdir(save_dir):
            if fname.startswith(student_id) and fname.endswith('.json'):
                with open(os.path.join(save_dir, fname)) as f:
                    sessions.append(json.load(f))
        return sorted(sessions, key=lambda s: s['session_start'])

    # ──────────────────────────────────────────────────────────────────────
    def print_summary(self):
        print('═' * 50)
        print(f'SESSION MEMORY — {self.student_id}')
        print('═' * 50)
        print(f'  Mastered topics : {self.mastered_topics}')
        print(f'  Weak topics     : {self.weak_topics}')
        print(f'  Confused terms  : {dict(list(self.confused_terms.items())[:5])}')
        print(f'  Total attempts  : {len(self.session_timeline)}')


print('✅ SessionMemoryStore class defined')

---
## Step 2 — Assessment Module

After the student completes a Socratic loop, prompt them with a clinical scenario and evaluate their reasoning.

In [ ]:
CLINICAL_SCENARIO_PROMPT = """You are an OT clinical reasoning assessor.
Given the anatomy/neuroscience concept and textbook context, write a ONE-PARAGRAPH
clinical OT scenario that requires the student to APPLY this concept.
The scenario should describe a patient with a realistic deficit or injury.
End with ONE clear question asking the student to reason through the clinical implication.
Keep it to 4-5 sentences. Do NOT give the answer.

Concept: {concept}
Textbook context: {context}"""


REASONING_EVAL_PROMPT = """You are evaluating an OT student's clinical reasoning.
Compare the student's reasoning against the textbook-grounded gold standard.

Concept: {concept}
Clinical question: {scenario}
Student's answer: {student_answer}
Textbook context (ground truth): {context}

Evaluate in this exact format:
ACCURACY: [Correct / Partially Correct / Incorrect]
STRENGTHS: [What they got right, 1-2 sentences]
GAPS: [What they missed or got wrong, 1-2 sentences]
MASTERY: [High / Medium / Low]
CLINICAL TAKEAWAY: [One sentence connecting this to OT practice]"""


class AssessmentModule:
    """
    Generates clinical OT scenarios and evaluates student reasoning
    against textbook-grounded gold standard answers.
    """

    def generate_scenario(self, concept: str, context: str) -> str:
        prompt = CLINICAL_SCENARIO_PROMPT.format(concept=concept, context=context[:1500])
        resp   = llm.invoke([HumanMessage(content=prompt)])
        return resp.content.strip()

    def evaluate_reasoning(self, concept: str, scenario: str,
                            student_answer: str, context: str) -> dict:
        prompt = REASONING_EVAL_PROMPT.format(
            concept=concept, scenario=scenario,
            student_answer=student_answer, context=context[:1500]
        )
        resp   = llm.invoke([HumanMessage(content=prompt)])
        raw    = resp.content.strip()

        # Parse structured output
        result = {'raw': raw}
        for field in ['ACCURACY', 'STRENGTHS', 'GAPS', 'MASTERY', 'CLINICAL TAKEAWAY']:
            import re
            match = re.search(rf'{field}:\s*(.+?)(?=\n[A-Z]|$)', raw, re.DOTALL)
            result[field.lower().replace(' ', '_')] = match.group(1).strip() if match else ''
        return result

    def generate_mastery_summary(self, student_id: str,
                                  memory: 'SessionMemoryStore') -> str:
        """Generate a full session mastery summary."""
        prompt = f"""You are an OT tutor generating a student mastery summary.

Student ID: {student_id}
Topics mastered (answered by Turn 2): {memory.mastered_topics}
Topics that needed full reveal (Turn 3): {memory.weak_topics}
Most confused terms: {memory.get_most_confused_terms(5)}
Wrong guesses by topic: {json.dumps(memory.wrong_guesses, indent=2)[:800]}

Write a 6-8 sentence mastery summary covering:
1. Overall performance this session
2. Strongest topics
3. Topics needing review
4. Specific misconceptions to address
5. Recommended next study focus for NBCOT prep"""
        resp = llm.invoke([HumanMessage(content=prompt)])
        return resp.content.strip()


assessor = AssessmentModule()
print('✅ AssessmentModule defined')

---
## Step 3 — Full 15-Minute Session Orchestrator

Combines memory + tutoring policy + assessment into a complete session flow.

In [ ]:
import time

class OTTutoringSession:
    """
    Orchestrates a full 15-minute Socratic OT tutoring session.
    Combines: retrieval → Socratic policy → memory → assessment → summary
    """

    SESSION_DURATION_MINUTES = 15
    MAX_TOPICS_PER_SESSION   = 4   # ~3-4 topics in 15 minutes

    def __init__(self, student_id: str = 'student_001'):
        self.memory    = SessionMemoryStore(student_id)
        self.assessor  = AssessmentModule()
        self.start_time = time.time()
        self.topics_covered = 0
        self.current_topic  = None
        self.current_masked = None
        self.current_turn   = 0    # turns within current Socratic loop
        self.history        = []   # LangChain messages
        self.phase          = 'RAPPORT'

    def _time_remaining(self) -> float:
        """Minutes remaining in the 15-minute session."""
        elapsed = (time.time() - self.start_time) / 60
        return max(0, self.SESSION_DURATION_MINUTES - elapsed)

    def _get_context(self, query: str) -> str:
        chunks = retrieve(query)
        return '\n\n'.join(c['text'] for c in chunks)

    def _build_proactive_opener(self) -> str:
        """If weak topics exist, build an opener that proactively revisits them."""
        weak = self.memory.get_weak_topics_for_revisit(n=1)
        if not weak:
            return ''
        topic = weak[0]
        return (f'Before we move on, I noticed you had some difficulty with "{topic}" earlier. '
                f"Let's revisit it briefly. ")

    def chat(self, user_input: str) -> str:
        """
        Process one student turn.
        Handles the full state machine: rapport → hint → clue → reveal → assess.
        """
        # ── Session time check ─────────────────────────────────────────────
        if self._time_remaining() < 1:
            return self._wrap_up_session()

        context = self._get_context(user_input)
        self.current_turn += 1

        # ── Phase dispatch ─────────────────────────────────────────────────
        if self.phase == 'RAPPORT':
            return self._handle_rapport(user_input, context)
        elif self.phase == 'HINT':
            return self._handle_hint(user_input, context)
        elif self.phase == 'CLUE':
            return self._handle_clue(user_input, context)
        elif self.phase == 'REVEAL':
            return self._handle_reveal(user_input, context)
        elif self.phase == 'ASSESS':
            return self._handle_assess(user_input, context)
        else:
            return self._wrap_up_session()

    def _call(self, system_prompt: str, user_msg: str) -> str:
        msgs = [SystemMessage(content=system_prompt)] + self.history[-6:]
        if user_msg:
            msgs.append(HumanMessage(content=user_msg))
        resp = llm.invoke(msgs)
        text = resp.content.strip()
        self.history.extend([HumanMessage(content=user_msg), AIMessage(content=text)])
        return text

    def _extract_masked(self, query: str, context: str) -> str:
        p = f'Identify the single key anatomy/neuroscience term the student is asking about. Return ONLY that term (2-5 words).\nQuestion: {query}\nContext: {context[:600]}'
        r = llm.invoke([HumanMessage(content=p)])
        return r.content.strip()

    def _handle_rapport(self, user_input: str, context: str) -> str:
        self.current_masked = self._extract_masked(user_input, context)
        self.current_topic  = user_input[:60]
        opener = self._build_proactive_opener()
        sys = f'You are a warm Socratic OT tutor. Build brief rapport about the student topic. {opener}Keep it to 2 sentences then ask a warm engagement question. Topic context: {context[:500]}'
        resp = self._call(sys, user_input)
        self.phase = 'HINT'
        return resp

    def _handle_hint(self, user_input: str, context: str) -> str:
        masked = self.current_masked
        sys = f'You are a Socratic OT tutor. STRICT RULE: Do NOT name or define "{masked}" directly. Ask ONE leading question using this context to help the student recall it.\nCONTEXT: {context[:1500]}\nMASKED ANSWER (never reveal): {masked}'
        resp = self._call(sys, user_input)
        if masked.lower() in resp.lower():
            resp = self._call(sys + '\nCRITICAL: Rewrite WITHOUT mentioning the answer.', '')
        self.phase = 'CLUE'
        return resp

    def _handle_clue(self, user_input: str, context: str) -> str:
        masked = self.current_masked
        # Check if student got it
        if masked.lower() in user_input.lower():
            self.memory.record_attempt(self.current_topic, user_input, True, self.current_turn)
            self.phase = 'ASSESS'
            return f'Exactly right! {masked} — well done. ' + self._transition_to_assess(context)

        self.memory.record_attempt(self.current_topic, user_input, False, self.current_turn)
        sys = f'You are a Socratic OT tutor. Give a MORE specific clue about "{masked}" using the context. Still do NOT name it directly. Student tried: "{user_input}".\nCONTEXT: {context[:1500]}\nMASKED ANSWER: {masked}'
        resp = self._call(sys, user_input)
        if masked.lower() in resp.lower():
            resp = self._call(sys + '\nRewrite clue WITHOUT stating the answer.', '')
        self.phase = 'REVEAL'
        return resp

    def _handle_reveal(self, user_input: str, context: str) -> str:
        masked = self.current_masked
        correct = masked.lower() in user_input.lower()
        self.memory.record_attempt(self.current_topic, user_input, correct, self.current_turn)
        self.memory.record_topic_outcome(self.current_topic,
                                          turns_to_correct=self.current_turn,
                                          needed_reveal=not correct)
        sys = f'You are a Socratic OT tutor. The student may or may not have gotten it. Now CONFIRM or CORRECT them. State the answer clearly: "{masked}". Give a textbook-grounded explanation and a brief OT clinical connection.\nCONTEXT: {context[:1500]}'
        resp = self._call(sys, user_input)
        self.phase = 'ASSESS'
        return resp + '\n\n' + self._transition_to_assess(context)

    def _transition_to_assess(self, context: str) -> str:
        scenario = self.assessor.generate_scenario(self.current_masked, context)
        self.current_scenario = scenario
        return '\n**Clinical Application Challenge:**\n' + scenario

    def _handle_assess(self, student_answer: str, context: str) -> str:
        eval_result = self.assessor.evaluate_reasoning(
            concept       = self.current_masked,
            scenario      = getattr(self, 'current_scenario', ''),
            student_answer= student_answer,
            context       = context
        )
        self.topics_covered += 1
        self.phase = 'RAPPORT' if self.topics_covered < self.MAX_TOPICS_PER_SESSION else 'WRAP'
        self.current_turn   = 0
        self.current_masked = None

        feedback = f"""
**Assessment Feedback:**
- Accuracy: {eval_result.get('accuracy', '')}
- Strengths: {eval_result.get('strengths', '')}
- Gaps: {eval_result.get('gaps', '')}
- Mastery: {eval_result.get('mastery', '')}
- Clinical Takeaway: {eval_result.get('clinical_takeaway', '')}
"""
        if self.phase == 'WRAP':
            return feedback + '\n' + self._wrap_up_session()
        return feedback + '\n\nReady for the next concept? What topic would you like to review next?'

    def _wrap_up_session(self) -> str:
        summary = self.assessor.generate_mastery_summary(self.memory.student_id, self.memory)
        self.memory.save()
        self.phase = 'DONE'
        return f'\n**Session Mastery Summary:**\n{summary}'


print('✅ OTTutoringSession orchestrator defined')

---
## Step 4 — Demo Session

In [ ]:
def run_full_session_demo():
    session = OTTutoringSession(student_id='demo_student')

    # Simulated conversation for 2 topics
    script = [
        # Topic 1: Median nerve
        'I want to review the nerve responsible for thumb opposition',
        'Is it the ulnar nerve?',       # HINT turn — wrong
        'Radial nerve?',                # CLUE turn — wrong
        'Median nerve!',                # REVEAL turn — correct
        'The patient cant pinch because the median nerve controls the thenar muscles,'
        ' so opposition and fine motor grip are lost.',  # ASSESS answer

        # Topic 2: Sarcomere
        'Now help me with how muscles contract at the microscopic level',
        'Something with filaments?',    # HINT
        'Actin and myosin sliding?',    # CLUE — correct early
        'The patient with ALS loses motor neurons so the actin-myosin cross bridges'
        ' cant form, causing progressive weakness and eventual paralysis.',  # ASSESS
    ]

    for i, msg in enumerate(script):
        print(f'\n{'─'*60}')
        print(f'[Turn {i+1} | Phase: {session.phase}]')
        print(f'Student: {msg}')
        response = session.chat(msg)
        print(f'Tutor:\n{response}')
        if session.phase == 'DONE':
            break

    print('\n' + '═'*60)
    print('SESSION MEMORY FINAL STATE:')
    session.memory.print_summary()


run_full_session_demo()

---
## Step 5 — Student Dashboard (Weak Spot Tracker)

In [ ]:
from collections import Counter

def generate_student_dashboard(student_id: str) -> dict:
    """
    Aggregates all past sessions for a student and returns a dashboard dict.
    Shows cumulative weak spots, mastered topics, and improvement over time.
    """
    history = SessionMemoryStore.load_student_history(student_id)

    if not history:
        print(f'No session history found for {student_id}')
        return {}

    # Aggregate across sessions
    all_weak      = []
    all_mastered  = []
    all_confused  = Counter()

    for session in history:
        all_weak.extend(session.get('weak_topics', []))
        all_mastered.extend(session.get('mastered_topics', []))
        for term, count in session.get('confused_terms', {}).items():
            all_confused[term] += count

    # Topics that appear more weak than mastered
    weak_freq     = Counter(all_weak)
    mastered_freq = Counter(all_mastered)
    priority_review = [
        topic for topic, cnt in weak_freq.most_common(10)
        if cnt > mastered_freq.get(topic, 0)
    ]

    dashboard = {
        'student_id':        student_id,
        'total_sessions':    len(history),
        'last_session':      history[-1]['session_start'],
        'mastered_topics':   list(set(all_mastered)),
        'weak_topics':       list(set(all_weak)),
        'priority_review':   priority_review[:5],
        'top_confused_terms': all_confused.most_common(5),
        'progress': {
            'sessions':    [s['session_id'] for s in history],
            'weak_counts': [len(s.get('weak_topics', [])) for s in history]
        }
    }

    # Print
    print(f"""{'═'*55}
STUDENT DASHBOARD — {student_id}
{'═'*55}
Total sessions      : {dashboard['total_sessions']}
Last session        : {dashboard['last_session']}
Topics mastered     : {len(dashboard['mastered_topics'])}
Weak topics         : {len(dashboard['weak_topics'])}
Priority for review : {dashboard['priority_review']}
Most confused terms : {[t for t,_ in dashboard['top_confused_terms']]}
{'═'*55}""")

    return dashboard


# Try loading (will show empty if no sessions saved yet)
dashboard = generate_student_dashboard('demo_student')

In [ ]:
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('PHASE 3 COMPLETE')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  Session Memory    : Weak topics, confused terms, mastery tracking')
print('  Proactive Revisit : Auto-surfaces past weak topics')
print('  Assessment Module : Clinical scenario generation + reasoning eval')
print('  Mastery Summary   : Per-topic + full session summary')
print('  Dashboard         : Cumulative weak-spot tracker across sessions')
print()
print('→ Next: Run 04_vlm_multimodal.ipynb')